# P17 — Modelos probabilísticos de difusión con eliminación de ruido

## 1. Título y paper

**Paper:** *Denoising Diffusion Probabilistic Models*  
**Autoría:** Jonathan Ho, Ajay Jain, Pieter Abbeel  
**Año y venue:** 2020 · arXiv:2006.11239 · NeurIPS 2020  
**Nivel:** L3 · **Motor:** `diffusion`  
**Ficha completa:** [`P17_diffusion`](../../papers/foundational/P17_diffusion/README.md)

**Hito:** La generación deja de ser un salto en la oscuridad: se aprende a deshacer, paso a paso, un proceso de ruido conocido.

- [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Las GAN generaban imágenes de calidad pero eran inestables de entrenar y colapsaban la diversidad; los VAE eran estables y producían muestras borrosas.
2. Ejecutar una implementación mínima de la propuesta: Un proceso directo que añade ruido gaussiano en T pasos con forma cerrada, y una red que aprende a predecir ese ruido para invertirlo.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Sohl-Dickstein et al. (2015), difusión no supervisada
- VAE y GAN como líneas base


## 4. Intuición

Destruir es fácil y reversible si sabes exactamente cómo destruiste. Añadir ruido gaussiano tiene fórmula cerrada; el modelo solo tiene que aprender a decir **cuánto ruido hay** en cada paso. Generar es deshacer ese camino.


## 5. Concepto mínimo

```text
Proceso directo (conocido, sin aprender):
    x_t = √ᾱ_t · x₀ + √(1−ᾱ_t) · ε,     ε ~ N(0, I),   ᾱ_t = Π_s (1−β_s)

Reconstrucción a partir de ε:
    x₀ = (x_t − √(1−ᾱ_t)·ε) / √ᾱ_t

Pérdida (forma simplificada del paper):
    L = E ‖ ε − ε_θ(x_t, t) ‖²        ← se predice el RUIDO, no la imagen
```


## 6. Código explicado

El motor calcula la trayectoria de ruido y reconstruye desde el paso más ruidoso con el ε correcto.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('diffusion', seed=7)['result']
for fila in r['trayectoria_de_ruido']:
    print(f"t={fila['t']:>2} · ᾱ={fila['alpha_barra']:.4f} · SNR={fila['snr']:>10.4f} · x_t={fila['x_t']}")
show(r['reconstruccion'])

## 7. Predicción antes de ejecutar

1. ¿Qué le pasa a la SNR al avanzar t: baja lineal o exponencialmente?
2. Con el ε exacto, ¿el error de reconstrucción será 0, ~1e-15 o ~0,01?
3. ¿En qué paso —el poco ruidoso o el muy ruidoso— duele más equivocarse en ε?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
import math
T = 20
betas = [1e-4 + (0.02 - 1e-4) * t / (T - 1) for t in range(T)]
ab, acum = [], 1.0
for b in betas:
    acum *= (1 - b)
    ab.append(acum)
print('amplificacion del error de epsilon = sqrt(1-ᾱ)/sqrt(ᾱ):')
for t in (0, 5, 10, 15, 19):
    print(f'  t={t:>2} → x{math.sqrt(1 - ab[t]) / math.sqrt(ab[t]):.2f}')

## 9. Salida interpretable

El factor de amplificación crece con `t`. Un mismo error en ε apenas se nota al principio y arruina la reconstrucción al final. **Por eso el muestreo va paso a paso** en vez de saltar del ruido puro a la imagen de una vez.


## 10. Comentario pedagógico

Fíjate en el cambio de marco: el problema generativo —difícil— se convirtió en un problema de **regresión supervisada** —fácil—, porque el par (entrada ruidosa, ruido) se puede fabricar gratis a partir de cualquier imagen. Ese truco es la contribución.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que el modelo predice la imagen limpia. Si predices x₀ directamente en un paso muy ruidoso, el objetivo es casi ruido puro.


In [ ]:
print('En t=19, ᾱ≈0.006: x_t es casi todo ruido.')
print('Predecir x0 desde ahi es adivinar; predecir ε es una tarea bien condicionada')
print('porque ε es exactamente lo que domina la señal en ese punto.')

## 12. Corrección

La parametrización correcta y su equivalencia:


In [ ]:
r = run_paper_lab('diffusion', seed=7)['result']['reconstruccion']
print('original                :', r['original'])
print('reconstruido desde ε    :', r['con_epsilon_correcto'])
print('error con ε correcto    :', r['error_con_epsilon_correcto'])
print('error con ε desviado 0.5:', r['error_con_epsilon_desviado_0_5'])

## 13. Desafío guiado

Cambia el planificador de β a uno coseno y compara cómo cae la SNR.


In [ ]:
import math
T = 20
lineal = [1e-4 + (0.02 - 1e-4) * t / (T - 1) for t in range(T)]
coseno = [min(0.999, 1 - math.cos((t + 1) / T * math.pi / 2) ** 2 / max(math.cos(t / T * math.pi / 2) ** 2, 1e-8)) for t in range(T)]
for nombre, betas in (('lineal', lineal), ('coseno', coseno)):
    acum = 1.0
    for b in betas:
        acum *= (1 - b)
    print(f'{nombre:<7} → ᾱ_final = {acum:.6f}')

## 14. Desafío autónomo

Implementa el muestreo inverso completo (DDPM ancestral) sobre datos 2D sintéticos con dos modos, entrenando una red pequeña para predecir ε. Mide si las muestras cubren ambos modos o colapsan en uno, y compáralo con lo que haría una GAN de tamaño similar.


## 15. Evidencia de aprendizaje

Guarda la trayectoria de SNR, la tabla de amplificación del error por paso y tu explicación de por qué se predice el ruido y no la imagen.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P17_diffusion/README.md) · evaluación formal: [`assessments/papers/P17_diffusion.md`](../../assessments/papers/P17_diffusion.md)


## 16. Cierre

La generación de imágenes ya tiene un objetivo estable y entrenable. Falta poder **decirle qué generar** con palabras: eso exige un espacio compartido entre imagen y texto.


## 17. Conexión con el siguiente hito

- P18
- modelos de imagen y vídeo a gran escala

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
